## RNN Prediction using training model.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [2]:
word_index = imdb.get_word_index() # this is a dictionary that maps words to their integer index in the dataset
reverse_word_index = {value: key for (key, value) in word_index.items()} # this is a dictionary that maps integer index to words

In [3]:
model = load_model('imdb_rnn_model.h5') # load the saved model
model.summary() # print the model summary to see the architecture of the model


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 500, 128)          1280000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               32896     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1313025 (5.01 MB)
Trainable params: 1313025 (5.01 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [4]:
model.get_weights() # this will give us the weights of the model, we can see the weights of each layer in the model, we can also see the shape of the weights, for example the embedding layer will have a weight matrix of shape (max_features, embedding_dim) which is (10000, 128) in our case, and the RNN layer will have a weight matrix of shape (embedding_dim + rnn_units, rnn_units) which is (128 + 128, 128) in our case, and the output layer will have a weight matrix of shape (rnn_units, 1) which is (128, 1) in our case

[array([[ 0.01176539,  0.04295864,  0.00733969, ...,  0.03216527,
          0.00998588,  0.04223846],
        [-0.04611028,  0.0110578 ,  0.02861524, ...,  0.02586152,
          0.0347213 ,  0.0684142 ],
        [-0.05498096, -0.01166169,  0.02690626, ..., -0.04968357,
         -0.02912013, -0.02800125],
        ...,
        [ 0.04044701, -0.04982829,  0.03988153, ...,  0.02561431,
          0.02687257,  0.04807526],
        [-0.02290842,  0.0594037 , -0.0033792 , ...,  0.00304661,
         -0.08841313, -0.0247809 ],
        [ 0.10232082, -0.13125166, -0.02912774, ...,  0.03326187,
          0.0983927 ,  0.08103424]], dtype=float32),
 array([[-0.06565193, -0.00114364, -0.02203948, ..., -0.01132407,
          0.11788853,  0.04717274],
        [ 0.11142533, -0.0045358 ,  0.01790856, ..., -0.0445373 ,
         -0.09867149,  0.07294885],
        [ 0.04016276,  0.10552766, -0.03019531, ..., -0.05690831,
         -0.075898  , -0.02829104],
        ...,
        [ 0.11017235, -0.16377649,  0.0

In [8]:
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review]) # this will decode the encoded review back to words, we subtract 3 from the index because the first 3 indices are reserved for special tokens (padding, start of sequence and unknown)

def preprocess_text(text):
    # this function will preprocess the text by converting it to lowercase, removing punctuation and splitting it into words
    word = text.lower().split() # convert to lowercase
    encoded_review = [word_index.get(w, 2) + 3 for w in word] # encode the words
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500) # pad the review to a fixed length of 500
    return padded_review

In [9]:
# prediction function

def predict_sentiment(review):
    encoded_review = preprocess_text(review) # preprocess the review
    prediction = model.predict(encoded_review) # make a prediction using the model
    sentiment = 'positive' if prediction[0][0] >= 0.5 else 'negative' # determine the sentiment based on the predicted score
    return sentiment,prediction[0][0] # return the predicted sentiment score (between 0 and 1)

In [ ]:
example_review = "This movie was fantastic! I really enjoyed it." # example review to test the prediction function
predicted_sentiment, predicted_score = predict_sentiment(example_review) # get the predicted sentiment and score
print(f"Review: {example_review}")
print(f"Predicted Sentiment: {predicted_sentiment}")
print(f"Predicted Score: {predicted_score}")

1/1 [==============================] - 0s 33ms/step
Review: Good
Predicted Sentiment: negative
Predicted Score: 0.27397817373275757
